# 导入各种库

In [6]:
# numpy 在1.24以后没有np.bool8这些了
import numpy as np
# 给被删掉的别名补上到内置类型
if not hasattr(np, 'object'):
    np.object = object
if not hasattr(np, 'bool'):
    np.bool = bool
if not hasattr(np, 'int'):
    np.int = int
if not hasattr(np, 'float'):
    np.float = float
if not hasattr(np, 'complex'):
    np.complex = complex
if not hasattr(np, 'bool8'):
    np.bool8 = np.bool_

In [7]:
# %matplotlib inline
import matplotlib, matplotlib.pyplot as plt
print(matplotlib.get_backend())
from backtrader_plotting import Bokeh
from backtrader_plotting.schemes import Tradimo
from IPython.display import display
import jupyterlab, sys
print(jupyterlab.__version__, " @ ", sys.executable)


module://matplotlib_inline.backend_inline


Loading BokehJS ...

4.4.6  @  /Library/Developer/CommandLineTools/usr/bin/python3


In [8]:
import backtrader as bt
import backtrader.indicators as btind # 导入策略分析模块
import pandas as pd
pd.set_option('display.max_rows', None)
import tushare as ts
import warnings
import datetime
warnings.filterwarnings("ignore")



In [9]:
from pathlib import Path
import sys
root = Path.cwd()
root

PosixPath('/Users/qingbin.zhuang/Personal/StockProject/Backtrader/Strategy')

In [10]:
# 引入自己开发的模块
from pathlib import Path
import sys
root = Path.cwd()
if not (root / 'common').exists():      # 如果当前目录没有 common/
    sys.path.insert(0, str(root.parent.parent))  # 把父目录放到 import 路径最前

from common.utils import get_engine, AShareCommission, AShareSizer
from DataFetch.stock_basic import *

# 从本地mysql取数据

In [11]:
engine = get_engine()

In [12]:

ts_code = '000001.SZ'
query = f"""
SELECT * FROM daily_kline
WHERE ts_code = '{ts_code}'
ORDER BY trade_date ASC
"""
df_sql1 = pd.read_sql(query, engine)
df_sql1['date'] = pd.to_datetime(df_sql1['trade_date'])

In [13]:
ts_code = '002594.SZ'
query = f"""
SELECT * FROM daily_kline
WHERE ts_code = '{ts_code}'
ORDER BY trade_date ASC
"""
df_sql2 = pd.read_sql(query, engine)
df_sql2['date'] = pd.to_datetime(df_sql2['trade_date'])

In [14]:
ts_code = '300857.SZ'
query = f"""
SELECT * FROM daily_kline
WHERE ts_code = '{ts_code}'
ORDER BY trade_date ASC
"""
df_sql3 = pd.read_sql(query, engine)
df_sql3['date'] = pd.to_datetime(df_sql3['trade_date'])

In [15]:
df_sql1.head()

,ts_code,trade_date,open,high,low,close,pre_close,change,pct_chg,vol,amount,date
0,000001.SZ,20100104,24.52,24.58,23.68,23.71,24.37,-0.66,-2.71,241923.0,580250.0,2010-01-04
1,000001.SZ,20100105,23.75,23.90,22.75,23.30,23.71,-0.41,-1.73,556500.0,1293480.0,2010-01-05
2,000001.SZ,20100106,23.25,23.25,22.72,22.90,23.30,-0.40,-1.72,412143.0,944454.0,2010-01-06
3,000001.SZ,20100107,22.90,23.05,22.40,22.65,22.90,-0.25,-1.09,355337.0,804166.0,2010-01-07
4,000001.SZ,20100108,22.50,22.75,22.35,22.60,22.65,-0.05,-0.22,288543.0,650667.0,2010-01-08


In [16]:
df_sql2.tail()

,ts_code,trade_date,open,high,low,close,pre_close,change,pct_chg,vol,amount,date
3471,002594.SZ,20251103,100.79,100.99,98.50,99.60,100.79,-1.19,-1.1807,551670.0,5472680.0,2025-11-03
3472,002594.SZ,20251104,99.00,99.54,97.71,97.80,99.60,-1.80,-1.8072,435017.0,4274200.0,2025-11-04
3473,002594.SZ,20251105,96.66,96.66,94.73,95.80,97.80,-2.00,-2.0450,587215.0,5617220.0,2025-11-05
3474,002594.SZ,20251106,95.58,97.75,95.40,97.52,95.80,1.72,1.7954,402004.0,3897720.0,2025-11-06
3475,002594.SZ,20251107,97.00,98.15,96.83,97.20,97.52,-0.32,-0.3281,322561.0,3145820.0,2025-11-07


# 简单策略demo

In [17]:
class TS_Data(bt.feeds.PandasData):
    # 要添加的线 (初始 'close', 'low', 'high', 'open', 'volume', 'openinterest', 'datetime')
    lines = ('pre_close', 'change', 'amount', 'extra') 
    # 设置 line 在数据源上的列位置
    # -1 表示自动按列名匹配数据, None表示不存在（datetime则表示在index）, string直接写数据中的列名
    params = (
        ('fromdate', datetime.datetime(2022,4,1)),
        ('todate', datetime.datetime(2022,8,8)),
        ('nullvalue', 0.0),
        ('dtformat', ('%Y-%m-%d')),
        ('datetime', "date"),
        ('open', "open"),
        ('high', "high"),
        ('low', "low"),
        ('close', "close"),
        ('pre_close', "pre_close"),
        ('change', "change"),
        ('openinterest', "pct_chg"),
        ('volume', "vol"),
        ('amount', "amount"),
        ('extra', -1)
    )
    # def _load(self):
    #     # 调用原始的 _load 方法
    #     super()._load()

    #     # 添加你的自定义逻辑
    #     # _load方法是在每一个数据点被加载时调用的，所以你可以在这里添加你的自定义逻辑。
    #     # 你可以访问self.p.dataname来获取原始的Pandas DataFrame，然后从中提取你需要的数据。
    #     self.lines.extra[0] = self.p.dataname['extra'][self._idx]

    #     return True

In [ ]:
# 连涨策略（A股）：价量连续4天上涨 -> 第5天以开盘价买入 buy_pct 资金 ->
#              直到“价格或成交量任一出现下降”的那天，以收盘价卖出
# 说明：
# - 第5天开盘买入：开启 cheat_on_open，并设置 broker.set_coo(True)，在 next_open() 下单确保按开盘成交
# - 当天收盘卖出：需要 broker.set_coc(True)，并在 next() 里下单

# 启动阶段（只执行一次）
# __init__()：策略初始化（这里建了 pending_order 字典等）。
# （如果你实现了 start() 也会在这之后调一次；本策略没写）
# 回测运行中：对每一根 bar（每天）
# next_open()：先在“开盘时刻”调用（因为 cerebro = bt.Cerebro(cheat_on_open=True) 开启了）。
# 你在这里做“第5天开盘买入”的判断并下单。
# next()：再在“收盘/本 bar 结束时刻”调用。
# 你在这里做“价或量下降则当日收盘卖出”的判断并下单（exectype=bt.Order.Close + broker.set_coc(True)）。
# notify_order(order)：不固定插在 next/next_open 的哪一行，但会在订单状态变化时被调用：
# 常见表现是：你在 next_open()/next() 里下单后，订单被 Submitted/Accepted，随后在成交点变成 Completed，触发一次 notify_order。
# 你看到的日志里 BUY_OPEN ... / SELL_CLOSE ... 就是 notify_order 打出来的。
class FourUpVolUpBuyOpenSellCloseOnDownStrategyV1(bt.Strategy):
    """
    - 连续4天价量严格上涨 -> 第5天开盘买入
    - 持仓后，价或量任一下降 -> 触发当天收盘卖出
    """

    params = dict(
        up_days=4,        # 连续上涨天数（价 & 量都必须严格递增）
        buy_pct=0.2,      # 每次买入使用可用资金比例
        lot=100,          # A股最小交易单位：100股
        buy_buffer=0.98,  # 用 close 估算 next open 时的安全折扣（防止跳空高开导致资金不足）
        printlog=True,
    )

    def __init__(self):
        self.pending_order = {d: None for d in self.datas}

        # 统计指标
        self.start_value = None
        self.end_value = None

        # 交易统计（以“平仓 trade”为粒度）
        self.gross_pnl = 0.0          # 未扣手续费总收益（trade.pnl 求和）
        self.net_pnl = 0.0            # 扣手续费后总收益（trade.pnlcomm 求和）
        self.fees_total = 0.0         # 手续费总额（gross - net）
        self.closed_trades = 0
        self.win_trades = 0

    def start(self):
        # 回测开始时的权益（含现金+持仓市值）
        self.start_value = float(self.broker.getvalue())

    def notify_trade(self, trade):
        # 每个 data 的交易闭合时回调
        if trade.isclosed:
            self.closed_trades += 1

            gross = float(getattr(trade, 'pnl', 0.0) or 0.0)
            net = float(getattr(trade, 'pnlcomm', 0.0) or 0.0)
            fees = gross - net

            self.gross_pnl += gross
            self.net_pnl += net
            self.fees_total += fees

            if net > 0:
                self.win_trades += 1

    def stop(self):
        # 回测结束时的权益
        self.end_value = float(self.broker.getvalue())

    def log(self, txt, dt=None):
        if not self.p.printlog:
            return
        if dt is None:
            dt = bt.num2date(self.datas[0].datetime[0])
        print(f"{dt.date()} {txt}")

    # 多 data 不对齐时（有的 data 还没开始，len(d)=0），backtrader 会先走 prenext。
    # 默认 prenext 是 pass，会导致策略看起来“不执行”。这里让它继续复用 next。
    def prenext(self):
        self.next()

    def nextstart(self):
        self.next()

    def next(self):
        # 先处理卖出（需要 coc=True 才能触发当日收盘成交）
        for d in self.datas:
            if self.pending_order.get(d) is not None:
                continue

            pos = self.getposition(d)
            if pos.size <= 0:
                continue
            if len(d) < 2:
                continue

            price_down = float(d.close[0]) < float(d.close[-1])
            vol_down = float(d.volume[0]) < float(d.volume[-1])
            if price_down or vol_down:
                # Cheat-On-Close：让 Market 单按“当根 close”成交
                #     开启：cerebro.broker.set_coc(True)（或等价配置）
                #     效果：你在第 N 根 bar 的 next() 里发的 Market 单，可以直接按第 N 根 bar 的 close撮合（官方明确说这是 “cheating”）
                # Cheat-On-Open：让你在开盘前下单、按“当根 open”成交
                #     开启：cerebro = bt.Cerebro(cheat_on_open=True) 并确保 broker coo 打开（Cerebro 默认会自动帮你 set_coo(True)，除非你关掉）
                #     下单位置：写在 next_open() 里
                #     成交价位：当根 bar 的 open（同一天同一根开盘价）
                self.broker.set_coc(True)
                # 在 Backtrader 里，bt.Order 常说的“类型”一般指 exectype（执行类型），常用这些：
                    # bt.Order.Market：市价单（下一次可成交时成交，回测里通常是下一根 bar 的开盘）
                    # bt.Order.Close：收盘单（按当日收盘价成交；通常需要 broker.set_coc(True) 才能当日收盘成交）
                    # bt.Order.Limit：限价单（达到/优于限价才成交）
                    # bt.Order.Stop：止损/止动单（触发价到达后变成市价单）
                    # bt.Order.StopLimit：止损限价单（触发后变成限价单）
                    # bt.Order.StopTrail：跟踪止损（止损价随价格有利方向移动）
                    # bt.Order.StopTrailLimit：跟踪止损限价
                # 另外补充两类也常被叫“类型”：
                    # 方向：order.isbuy() / order.issell()
                    # 状态：Submitted / Accepted / Completed / Canceled / Margin / Rejected 等（用 order.status / order.getstatusname() 看）
                self.pending_order[d] = self.sell(data=d, size=pos.size, exectype=bt.Order.Close)

        # 再处理买入：在“连续上涨 up_days 天”确认信号，并下单让其在“下一根bar开盘”成交
        for d in self.datas:
            if self.pending_order.get(d) is not None:
                continue
            if self.getposition(d).size != 0:
                continue

            n = int(self.p.up_days)
            if n < 2:
                continue
            if len(d) < n:
                continue

            # 最近 n 天（含今天）价量严格递增：close[0] > close[-1] > ... > close[-(n-1)]
            cond_price = all(float(d.close[-k]) > float(d.close[-k - 1]) for k in range(0, n - 1))
            cond_vol = all(float(d.volume[-k]) > float(d.volume[-k - 1]) for k in range(0, n - 1))
            if not (cond_price and cond_vol):
                continue

            price_est = float(d.close[0])
            if price_est <= 0:
                continue

            cash = float(self.broker.getcash())
            target_cash = cash * float(self.p.buy_pct)
            target_cash *= float(self.p.buy_buffer)

            raw_size = int(target_cash / price_est)
            lot = int(self.p.lot)
            size = (raw_size // lot) * lot
            if size <= 0:
                continue

            # 确保买单不会被 coc 撮合到当日收盘；默认 market 单会在下一根bar开盘成交
            self.broker.set_coc(False)
            self.pending_order[d] = self.buy(data=d, size=size)

    def notify_order(self, order):
        if order.status in [order.Submitted, order.Accepted]:
            return

        d = order.data
        dt = bt.num2date(d.datetime[0])

        if order.status == order.Completed:
            if order.isbuy():
                self.log(
                    f"BUY_OPEN   {d._name} size={order.executed.size} price={order.executed.price:.2f}",
                    dt=dt,
                )
            else:
                self.log(
                    f"SELL_CLOSE {d._name} size={order.executed.size} price={order.executed.price:.2f}",
                    dt=dt,
                )
        elif order.status in [order.Canceled, order.Margin, order.Rejected]:
            self.log(f"ORDER_FAIL {d._name} status={order.getstatusname()}", dt=dt)

        if self.pending_order.get(d) == order:
            self.pending_order[d] = None


# === 运行回测（两只股票） ===

# 关键：
# - runonce=False 强制逐bar执行（多 data 未对齐时，配合策略里的 prenext/nextstart 才能“有数据的先跑”）
# - preload=False（可选）避免一次性预加载导致的同步歧义，便于调试
# NOTE: V1 策略不再用 next_open，因此不需要 cheat_on_open
cerebro = bt.Cerebro(runonce=False, preload=False)

# 初始资金 100000
cerebro.broker.setcash(500000.0)

# 交易费用 + 滑点（A股：双边佣金 + 卖出印花税 + 最低佣金；可选沪市过户费）
comminfo = AShareCommission(
    commission=0.0003,      # 券商佣金 0.03%（买卖双边）
    stamp_duty=0.0005,      # 卖出印花税 0.05%（仅卖出）
    min_commission=5.0,     # 单笔最低佣金 5 元（仅对佣金部分生效）
    transfer_fee=0.0        # 如为上证 .SH 可设为 0.00002；深证一般为 0
)
cerebro.broker.addcommissioninfo(comminfo)

# 滑点（按成交价比例）
cerebro.broker.set_slippage_perc(perc=0.0001)

# 默认关闭 coc，避免“在 next() 下的买单”被撮合到当日收盘
# （策略会在需要“触发当日收盘卖出”时临时开启 coc）
cerebro.broker.set_coc(False)

# 加载多只股票数据：按“每只股票的最早交易日”排序后再 adddata
# 这样 data0 会是最早开始的那只，不会被“新股/最近才有数据的股票”拖到很晚才开始

ts_data_info = {
    '000001.SZ': '平安银行',
    '300857.SZ': '协创数据',
    '002594.SZ': '比亚迪',
}

feeds = []  # (first_date, ts_code, name, df_sql)
for ts_code, name in ts_data_info.items():
    query = f"""
    SELECT * FROM daily_kline
    WHERE ts_code = '{ts_code}'
    ORDER BY trade_date ASC
    """
    df_sql = pd.read_sql(query, engine)
    df_sql['date'] = pd.to_datetime(df_sql['trade_date'])
    first_date = df_sql['date'].min()
    feeds.append((first_date, ts_code, name, df_sql))

feeds.sort(key=lambda x: x[0])
print('Data add order (earliest -> latest):', [(x[1], str(x[0].date())) for x in feeds])

for first_date, ts_code, name, df_sql in feeds:
    # 每只 data 从自己的最早日期开始（避免 leading empty bars 的各种同步歧义）
    data = TS_Data(dataname=df_sql, fromdate=first_date.to_pydatetime(), todate=datetime.datetime(2025, 8, 8))
    cerebro.adddata(data, name=name)


# 策略
cerebro.addstrategy(FourUpVolUpBuyOpenSellCloseOnDownStrategyV1, up_days=4, buy_pct=0.2, printlog=True)

print('Starting Portfolio Value: %.2f' % cerebro.broker.getvalue())
results = cerebro.run()
print('Final Portfolio Value: %.2f' % cerebro.broker.getvalue())

# === 统计输出（扣费前/后收益、手续费、交易次数、胜率）===
strat = results[0]
start_value = float(strat.start_value or 0.0)
end_value = float(strat.end_value or cerebro.broker.getvalue())

# 以平仓 trade 为粒度：
# - gross_pnl: 未扣费收益（trade.pnl）
# - net_pnl:   扣费后收益（trade.pnlcomm）
# - fees_total: 手续费（gross - net）
gross_pnl = float(strat.gross_pnl)
net_pnl_trades = float(strat.net_pnl)
fees_total = float(strat.fees_total)

# 以账户权益为准的“最终净收益”（含所有已实现/未实现）
net_pnl_equity = end_value - start_value

gross_ret = (gross_pnl / start_value) if start_value else 0.0
net_ret_equity = (net_pnl_equity / start_value) if start_value else 0.0

trade_cnt = int(strat.closed_trades)
win_cnt = int(strat.win_trades)
win_rate = (win_cnt / trade_cnt) if trade_cnt else 0.0

print('\n=== Performance Summary ===')
print(f'Start value: {start_value:,.2f}')
print(f'End value:   {end_value:,.2f}')
print(f'Gross PnL (before fees, closed trades): {gross_pnl:,.2f} ({gross_ret:.2%})')
print(f'Net PnL (after fees, equity-based):     {net_pnl_equity:,.2f} ({net_ret_equity:.2%})')
print(f'Fees total (estimated from closed trades): {fees_total:,.2f}')
print(f'Trades closed: {trade_cnt}, Wins: {win_cnt}, Win rate: {win_rate:.2%}')


Data add order (earliest -> latest): [('000001.SZ', '2010-01-04'), ('002594.SZ', '2011-06-30'), ('300857.SZ', '2020-07-27')]
Starting Portfolio Value: 100000.00
2010-03-03 BUY_OPEN   平安银行 size=800 price=23.11
2010-03-04 SELL_CLOSE 平安银行 size=-800 price=23.10
2010-11-04 BUY_OPEN   平安银行 size=1000 price=18.93
2010-11-05 SELL_CLOSE 平安银行 size=-1000 price=18.93
2011-03-04 BUY_OPEN   平安银行 size=1100 price=16.45
2011-03-07 SELL_CLOSE 平安银行 size=-1100 price=16.74
2011-03-30 BUY_OPEN   平安银行 size=1100 price=16.48
2011-03-31 SELL_CLOSE 平安银行 size=-1100 price=16.08
2012-01-12 BUY_OPEN   比亚迪 size=800 price=23.98
2012-01-13 SELL_CLOSE 比亚迪 size=-800 price=23.87
2012-02-29 BUY_OPEN   平安银行 size=1100 price=17.27
2012-03-01 SELL_CLOSE 平安银行 size=-1100 price=17.23
2012-03-08 BUY_OPEN   比亚迪 size=700 price=27.55
2012-03-09 SELL_CLOSE 比亚迪 size=-700 price=27.39
2012-08-01 BUY_OPEN   平安银行 size=1200 price=15.08
2012-08-02 SELL_CLOSE 平安银行 size=-1200 price=15.16
2012-08-07 BUY_OPEN   比亚迪 size=1200 price=15.50
2012-08-0

In [79]:
df_sql[(df_sql.date >= '2024-10-29') & (df_sql.date <= '2025-10-22')]

,ts_code,trade_date,open,high,low,close,pre_close,change,pct_chg,vol,amount,date
1031,300857.SZ,20241029,77.14,77.40,74.01,74.55,77.90,-3.35,-4.3004,159780.0,1208200.0,2024-10-29
1032,300857.SZ,20241030,74.90,75.98,72.17,73.35,74.55,-1.20,-1.6097,127948.0,948339.0,2024-10-30
1033,300857.SZ,20241031,73.21,75.65,72.00,73.30,73.35,-0.05,-0.0682,141743.0,1045800.0,2024-10-31
1034,300857.SZ,20241101,72.81,73.26,69.60,70.36,73.30,-2.94,-4.0109,125481.0,893619.0,2024-11-01
1035,300857.SZ,20241104,70.37,75.28,70.30,75.28,70.36,4.92,6.9926,162122.0,1191200.0,2024-11-04
1036,300857.SZ,20241105,75.29,81.55,74.71,81.12,75.28,5.84,7.7577,201007.0,1593340.0,2024-11-05
1037,300857.SZ,20241106,81.00,86.23,79.24,83.96,81.12,2.84,3.5010,201774.0,1661410.0,2024-11-06
1038,300857.SZ,20241107,82.99,91.93,82.53,91.78,83.96,7.82,9.3140,194281.0,1702960.0,2024-11-07
1039,300857.SZ,20241108,92.00,97.48,89.87,91.50,91.78,-0.28,-0.3051,179281.0,1673270.0,2024-11-08
1040,300857.SZ,20241111,90.01,94.03,88.70,92.36,91.50,0.86,0.9399,138690.0,1280820.0,2024-11-11


In [ ]:
cerebro.plot(Bokeh(style='bar')) 

[[<backtrader_plotting.bokeh.bokeh.FigurePage at 0x1137a8160>]]